In [1]:
# ============================================================================
# APOLLO NLP ABLATION NOTEBOOK – DistilBERT on AG News
# Researcher: Dr. Ali Zeydi Abdian
# Addresses EBM #3, R2 #1, R2 #3, R2 #4, R2 #5, R1 #6, R1 #7, R1 #11, R1 #12
# QWEN & EXPERT FIXES: Fixed Fatal Statistical Paradox (d_z vs p-value), Cleaned Metrics,
# Added Pure_AdamW/Pure_Lion baselines, fixed standard Lion update formula,
# completed dynamic subplot generation for Raw Cosine Similarity,
# integrated Auto-ZIP export for Kaggle output, ADDED LIVE EPOCH LOGGING,
# UPGRADED TO DistilBERT for ultra-fast, data-efficient ablation,
# FIXED MISMATCHED SIZES ERROR for offline loading,
# INTEGRATED Multi-GPU DataParallel scaling,
# and FIXED FATAL EPOCH-BOUNDARY CRASH via Upfront Pre-Tokenization.
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import os, time, warnings, json, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import ttest_rel, t as t_dist
from statsmodels.stats.multitest import multipletests

# ============================================================================
# 🛡️ ENVIRONMENT & REPRODUCIBILITY
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false" 

warnings.filterwarnings("ignore")
try:
    torch.set_float32_matmul_precision('high')
except:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Device Verified: {device}", flush=True)

# Increased to 3 seeds for Statistical Significance
ABLATION_SEEDS = [42, 123, 777]  
SENSITIVITY_SEED = 42           

# Dataset & model paths (Kaggle offline)
AG_NEWS_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset'
DISTILBERT_PATH = '/kaggle/input/datasets/alkanerturan/distilbert'

# Output directory
RESULTS_DIR = "/kaggle/working/results_apollo_nlp_ablation"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "ablation_checkpoint.json")

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================
NLP_EPOCHS = 3
BATCH_SIZE = 128            # Saturated for dual T4 GPUs
MAX_LENGTH = 128
GRADIENT_CLIP_VALUE = 1.0
WARMUP_RATIO = 0.1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 🧠 APOLLO OPTIMIZER
# ============================================================================
class Apollo(Optimizer):
    def __init__(self, params, lr=3e-5, betas=(0.9, 0.999, 0.9), eps=1e-8,
                 weight_decay=0.01, mode='standard', total_steps=None):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        self.mode = mode
        self.total_steps = total_steps
        self.current_step = 0
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        self.current_step += 1
        for group in self.param_groups:
            beta1, beta2, beta3 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['update_agreement'] = torch.zeros((), device=p.device)
                    state['last_cs'] = 0.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1

                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])

                lion_c = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                u_lion = torch.sign(lion_c)

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = exp_avg / bc1
                v_hat = exp_avg_sq / bc2
                denom = v_hat.sqrt().add_(group['eps'])
                u_adam = m_hat / denom

                adam_norm = u_adam.norm(p=2)
                lion_norm = u_lion.norm(p=2)
                if lion_norm > 0:
                    u_lion = u_lion * (adam_norm / lion_norm)

                cs_raw = F.cosine_similarity(u_adam.flatten(), u_lion.flatten(), dim=0, eps=1e-10)
                state['last_cs'] = cs_raw.item()

                if self.mode == 'pure_adamw':
                    gamma = 0.0
                elif self.mode == 'pure_lion':
                    gamma = 1.0
                elif self.mode == 'linear':
                    if self.total_steps is not None:
                        progress = self.current_step / self.total_steps
                        gamma = max(0.0, min(1.0, progress))
                    else:
                        gamma = 0.5
                elif self.mode == 'no_cosine':
                    gamma = 0.5
                else: 
                    state['update_agreement'].mul_(beta3).add_(cs_raw, alpha=1 - beta3)
                    bc3 = 1 - beta3 ** state['step']
                    agreement_hat = state['update_agreement'] / bc3
                    gamma = (agreement_hat + 1.0) / 2.0

                p.add_((1 - gamma) * u_adam + gamma * u_lion, alpha=-group['lr'])
        return loss

# ============================================================================
# 📂 DATA LOADER (UPFRONT RAM PRE-TOKENIZATION FOR MAX SPEED & STABILITY)
# ============================================================================
def load_agnews_data(tokenizer, max_length=128):
    train_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'train.csv'))
    test_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'test.csv'))
    
    for df in [train_df, test_df]:
        if 'Class Index' in df.columns:
            df.rename(columns={'Class Index': 'label'}, inplace=True)
        if 'Description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'Description': 'text'}, inplace=True)
        elif 'description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'description': 'text'}, inplace=True)
        if 'text' not in df.columns:
            label_col = 'label' if 'label' in df.columns else df.columns[0]
            df['text'] = df[[c for c in df.columns if c != label_col]].astype(str).agg(' '.join, axis=1)
            df.rename(columns={label_col: 'label'}, inplace=True)
        if df['label'].min() == 1:
            df['label'] -= 1

    print("⚡ Pre-tokenizing dataset into RAM for maximum iteration speed...", flush=True)
    
    # Pre-tokenize all data at once
    train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding='max_length', max_length=max_length)
    val_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding='max_length', max_length=max_length)

    class PreTokenizedDataset(Dataset):
        def __init__(self, encodings, labels):
            self.input_ids = encodings['input_ids']
            self.attention_mask = encodings['attention_mask']
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return {
                'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
                'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)
            }

    train_ds = PreTokenizedDataset(train_encodings, train_df['label'].tolist())
    val_ds = PreTokenizedDataset(val_encodings, test_df['label'].tolist())
    
    # Bypass all multiprocessing (num_workers=0) entirely. 
    # With upfront tokenization, CPUs are practically idle during training anyway.
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    return train_loader, val_loader

# ============================================================================
# 🔄 TRAINING LOOP
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, scheduler, num_epochs, use_amp=True):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_loss = float('inf')
    best_state = None
    patience = 5
    no_improve = 0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    raw_cosine_hist = []
    gamma_hist = []

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs.logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.step()

            scheduler.step()
            preds = outputs.logits.argmax(dim=1)
            tr_loss += loss.item() * labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                val_loss += loss.item() * labels.size(0)
                val_correct += outputs.logits.argmax(1).eq(labels).sum().item()
                val_total += labels.size(0)

        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if isinstance(optimizer, Apollo):
            if optimizer.param_groups and optimizer.param_groups[0]['params']:
                p0 = optimizer.param_groups[0]['params'][0]
                state = optimizer.state.get(p0, {})
                if 'last_cs' in state:
                    raw_cosine_hist.append(state['last_cs'])
                if optimizer.mode == 'standard':
                    if 'update_agreement' in state:
                        ag = state['update_agreement'].item()
                        gamma = (ag + 1) / 2.0
                        gamma_hist.append(gamma)
                    else:
                        gamma_hist.append(0.5)
                elif optimizer.mode == 'linear':
                    gamma = min(1.0, optimizer.current_step / optimizer.total_steps)
                    gamma_hist.append(gamma)
                elif optimizer.mode == 'pure_adamw':
                    gamma_hist.append(0.0)
                elif optimizer.mode == 'pure_lion':
                    gamma_hist.append(1.0)
                else: 
                    gamma_hist.append(0.5)
        else:
            raw_cosine_hist.append(0.0)
            gamma_hist.append(0.0)

        # LIVE PROGRESS PRINT
        print(f"      🔹 Epoch {epoch+1}/{num_epochs} | Val Acc: {history['val_acc'][-1]:.4f} | Time: {epoch_time:.0f}s", flush=True)

        if history['val_loss'][-1] < best_val_loss - 1e-4:
            best_val_loss = history['val_loss'][-1]
            no_improve = 0
            # VRAM LEAK FIX: Explicitly push copied weights to CPU to prevent memory bloat over multiple epochs
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            
        if no_improve >= patience and epoch >= patience:
            if best_state:
                # Ensure the model returns to the proper device after loading the CPU state dictates
                model.load_state_dict(best_state)
                model = model.to(device)
            break

    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.vstack(all_probs)

    acc = float((all_preds == all_targets).mean())
    f1 = float(f1_score(all_targets, all_preds, average='macro', zero_division=0))
    try:
        auc_val = float(roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'))
    except:
        auc_val = 0.0
    mcc = float(matthews_corrcoef(all_targets, all_preds))
    avg_epoch_time = float(np.mean(history['epoch_time']))

    return {
        'test_acc': acc,
        'test_f1': f1,
        'test_auc': auc_val,
        'test_mcc': mcc,
        'avg_epoch_time': avg_epoch_time,
        'history': history,
        'raw_cosine_hist': raw_cosine_hist,
        'gamma_hist': gamma_hist
    }

# ============================================================================
# 📊 STATISTICAL ANALYSIS & CHECKPOINTING
# ============================================================================
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                return json.load(f)
        except json.JSONDecodeError:
            pass
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results, f, indent=4)

def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        t_val = t_dist.ppf((1 + cl) / 2, max(n - 1, 1))
        margin = t_val * std / np.sqrt(n) if n > 0 else 0
        return float(mean - margin), float(mean + margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    alpha = (1 - cl) / 2
    return float(np.percentile(boot, 100 * alpha)), float(np.percentile(boot, 100 * (1 - alpha)))

def compute_statistics(multi_results):
    metrics = ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs:
            continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs]
            mean_v = float(np.mean(vals))
            std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_v
            agg[opt][f'{m}_std'] = std_v
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_v / mean_v * 100) if mean_v > 0 else 0)
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(results, baseline='Apollo_β3=0.9'):
    if baseline not in results or len(results[baseline]) < 2:
        return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig = {}
    pvals = []
    basenames = []
    
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2:
            continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals) != len(vals):
            continue
            
        basenames.append(opt)
        
        diffs = base_vals - vals
        mean_diff = float(np.mean(diffs))
        std_diff = float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z = 0.0
            p_raw = 1.0 if abs(mean_diff) < 1e-8 else 0.0
        else:
            d_z = float(mean_diff / std_diff)
            try:
                _, p_raw = ttest_rel(base_vals, vals)
            except:
                p_raw = 1.0
                
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_diff, 'p_raw': float(p_raw), 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(basenames):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = '***' if p_corr[i] < 0.001 else '**' if p_corr[i] < 0.01 else '*' if p_corr[i] < 0.05 else 'ns'
    return sig

# ============================================================================
# 📄 LATEX & PLOTTING FUNCTIONS
# ============================================================================
def ablation_performance_table(agg, ci, caption, label, filename):
    mc = [{'d':'Acc','k':'test_acc','f':'.4f'}, {'d':'F1','k':'test_f1','f':'.4f'},
          {'d':'AUC','k':'test_auc','f':'.4f'}, {'d':'MCC','k':'test_mcc','f':'.4f'},
          {'d':'Time(s)','k':'avg_epoch_time','f':'.2f'}]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
        r"{\bf Config} & " + " & ".join([f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"
    ]
    order = ['Pure_AdamW', 'Pure_Lion', 'Apollo_β3=0', 'Apollo_β3=0.9', 'Apollo_β3=0.99', 'Apollo_Fixedγ', 'Apollo_Linear']
    for opt in order:
        if opt not in agg: continue
        row = [opt.replace('_', ' ')]
        for m in mc:
            k, f = m['k'], m['f']
            mean, std = agg[opt][f'{k}_mean'], agg[opt][f'{k}_std']
            low, high = ci[opt][k]
            row.append(f"{mean:{f}} $\\pm$ {std:{f}} \\; [{low:{f}}, {high:{f}}]")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f:
        f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\begin{tabular}{lccccc}", r"\toprule",
        r"\textbf{Comparison vs $\beta_3$=0.9} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\",
        r"\midrule"
    ]
    for opt, r in sig.items():
        diff_str = f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$"
        lines.append(f"{opt.replace('_',' ')} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f:
        f.write("\n".join(lines))

def variance_table(var, cvs, caption, label, filename):
    metrics_show = [('test_acc', 'Accuracy'), ('test_f1', 'F1'), ('test_auc', 'AUC')]
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\begin{tabular}{l" + "cc" * len(metrics_show) + "}",
        r"\toprule",
        r"\multirow{2}{*}{\bf Config} " + "".join([f"& \\multicolumn{{2}}{{c}}{{\\bf {disp}}}" for _, disp in metrics_show]) + r" \\",
        r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}",
        r" & " + " & ".join([r"$\sigma^2$ & CV\%"] * len(metrics_show)) + r" \\",
        r"\midrule"
    ]
    order = ['Pure_AdamW', 'Pure_Lion', 'Apollo_β3=0', 'Apollo_β3=0.9', 'Apollo_β3=0.99', 'Apollo_Fixedγ', 'Apollo_Linear']
    for opt in order:
        if opt not in var:
            continue
        row = [opt.replace('_', ' ')]
        for key, _ in metrics_show:
            v = var[opt][f'{key}_var']
            if v < 1e-6:
                row.append(r"$<10^{-6}$")
            else:
                row.append(f"{v:.2e}")
            row.append(f"{cvs[opt][f'{key}_cv']:.2f}\\%")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f:
        f.write("\n".join(lines))

def plot_ablation_dynamics(ablation_results, title, save_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = plt.cm.tab10.colors
    configs = list(ablation_results.keys())
    
    # 1. Validation Accuracy
    ax = axes[0]
    for i, cfg in enumerate(configs):
        runs = ablation_results[cfg]
        if not runs: continue
        accs = [r['history']['val_acc'] for r in runs]
        max_len = max(len(a) for a in accs)
        padded = [np.pad(a, (0, max_len - len(a)), constant_values=np.nan) for a in accs]
        mean_acc = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_acc, label=cfg.replace('_', ' '), color=colors[i % len(colors)], linewidth=2)
    ax.set_title('Validation Accuracy', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=9)

    # 2. Gamma Weight
    ax = axes[1]
    for i, cfg in enumerate(configs):
        runs = ablation_results[cfg]
        if not runs: continue
        gammas = [r['gamma_hist'] for r in runs]
        max_len = max(len(g) for g in gammas)
        padded = [np.pad(g, (0, max_len - len(g)), constant_values=np.nan) for g in gammas]
        mean_g = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_g, label=cfg.replace('_', ' '), color=colors[i % len(colors)], linewidth=2)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.8)
    ax.set_title(r'Blending Weight $\gamma$', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)

    # 3. Raw Cosine Similarity
    ax = axes[2]
    for i, cfg in enumerate(configs):
        if 'Apollo' not in cfg and cfg != 'Pure_Lion': continue 
        runs = ablation_results[cfg]
        if not runs: continue
        cosines = [r['raw_cosine_hist'] for r in runs]
        if not any(cosines): continue
        max_len = max(len(c) for c in cosines if c)
        if max_len == 0: continue
        padded = [np.pad(c, (0, max_len - len(c)), constant_values=np.nan) for c in cosines if c]
        mean_c = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_c, label=cfg.replace('_', ' '), color=colors[i % len(colors)], alpha=0.8)
    ax.axhline(0.0, color='black', linestyle='-', alpha=0.5)
    ax.set_title('Raw Update Agreement (Cosine Similarity)', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# 📦 FINAL ZIP EXPORT
# ============================================================================
def create_final_zip():
    zip_path = "/kaggle/working/ablation_results.zip"
    base_folder = os.path.basename(RESULTS_DIR)
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(CHECKPOINT_FILE):
            zipf.write(CHECKPOINT_FILE, arcname=os.path.join(base_folder, os.path.basename(CHECKPOINT_FILE)))
        
        for sub_dir in ["plots", "latex_tables"]:
            dir_path = os.path.join(RESULTS_DIR, sub_dir)
            if os.path.isdir(dir_path):
                for root, dirs, files in os.walk(dir_path):
                    for file in files:
                        full_path = os.path.join(root, file)
                        arcname = os.path.join(base_folder, sub_dir, file)
                        zipf.write(full_path, arcname=arcname)
                        
    print(f"\n📦 Final results safely archived at: {zip_path}", flush=True)

# ============================================================================
# 🧪 ABLATION EXPERIMENTS WITH SMART CHECKPOINTING
# ============================================================================
def run_ablation_study(tokenizer):
    print("\n=== NLP ABLATION STUDY ===")
    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH)
    total_steps = NLP_EPOCHS * len(train_loader)

    configs = {
        'Pure_AdamW': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, mode='pure_adamw', total_steps=total_steps),
        'Pure_Lion': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, mode='pure_lion', total_steps=total_steps),
        'Apollo_β3=0': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, betas=(0.9,0.999,0.0), total_steps=total_steps),
        'Apollo_β3=0.9': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, betas=(0.9,0.999,0.9), total_steps=total_steps),
        'Apollo_β3=0.99': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, betas=(0.9,0.999,0.99), total_steps=total_steps),
        'Apollo_Fixedγ': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, mode='no_cosine', total_steps=total_steps),
        'Apollo_Linear': lambda p: Apollo(p, lr=3e-5, weight_decay=0.01, mode='linear', total_steps=total_steps)
    }
    
    results = load_checkpoint()
    for name in configs:
        if name not in results:
            results[name] = []

    for seed in ABLATION_SEEDS:
        for cfg_name, cfg_fn in configs.items():
            if len([r for r in results[cfg_name] if r.get('seed') == seed]) > 0:
                print(f"Skipping {cfg_name} seed={seed} (Already completed)", flush=True)
                continue

            set_seed(seed)
            print(f"\n🔥 Running {cfg_name} with seed={seed}...", flush=True)
            
            model = DistilBertForSequenceClassification.from_pretrained(
                DISTILBERT_PATH, 
                num_labels=4, 
                local_files_only=True,
                ignore_mismatched_sizes=True
            ).to(device)
            
            if torch.cuda.device_count() > 1:
                print(f"🚀 Utilizing {torch.cuda.device_count()} GPUs with DataParallel!", flush=True)
                model = nn.DataParallel(model)
            
            optimizer = cfg_fn(model.parameters())
            scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
            
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, NLP_EPOCHS)
            res['seed'] = seed
            print(f"   ✔️ {cfg_name} seed={seed} Final Acc={res['test_acc']:.4f}", flush=True)
            
            results[cfg_name].append(res)
            save_checkpoint(results)
            
    return results

# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("APOLLO NLP ABLATION NOTEBOOK – Statistically Robust & Optimized")
    print("=" * 80)

    tokenizer = DistilBertTokenizerFast.from_pretrained(DISTILBERT_PATH, local_files_only=True)
    ablation_results = run_ablation_study(tokenizer)
    
    abl_agg, abl_var, abl_cvs, abl_ci = compute_statistics(ablation_results)
    ablation_performance_table(abl_agg, abl_ci, r"Ablation Study (Pure Baselines, $\beta_3$, Fixed $\gamma$, Linear) - DistilBERT on AG News",
                                "tab:nlp_ablation_perf", "t1_ablation_perf.tex")
    
    abl_sig = statistical_tests(ablation_results, baseline='Apollo_β3=0.9')
    stats_table(abl_sig, r"Statistical Significance in Ablation (vs $\beta_3$=0.9, Paired T-Test) - DistilBERT on AG News",
                "tab:nlp_ablation_stats", "t2_ablation_stats.tex")
                
    variance_table(abl_var, abl_cvs, r"Stability Analysis (Variance \& CV) - DistilBERT on AG News",
                   "tab:nlp_ablation_var", "t3_ablation_var.tex")
    
    plot_ablation_dynamics(ablation_results, "NLP Ablation Dynamics (DistilBERT on AG News)", "ablation_dynamics.png")

    create_final_zip()
    print("\n✅ NLP ABLATION COMPLETE. Checkpoints, Results and LaTeX tables saved.")

🚀 Execution Device Verified: cuda
APOLLO NLP ABLATION NOTEBOOK – Statistically Robust & Optimized

=== NLP ABLATION STUDY ===
⚡ Pre-tokenizing dataset into RAM for maximum iteration speed...

🔥 Running Pure_AdamW with seed=42...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9297 | Time: 272s
      🔹 Epoch 2/3 | Val Acc: 0.9364 | Time: 273s
      🔹 Epoch 3/3 | Val Acc: 0.9362 | Time: 272s
   ✔️ Pure_AdamW seed=42 Final Acc=0.9362

🔥 Running Pure_Lion with seed=42...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9289 | Time: 273s
      🔹 Epoch 2/3 | Val Acc: 0.9376 | Time: 272s
      🔹 Epoch 3/3 | Val Acc: 0.9397 | Time: 272s
   ✔️ Apollo_Fixedγ seed=777 Final Acc=0.9397

🔥 Running Apollo_Linear with seed=777...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9293 | Time: 272s
      🔹 Epoch 2/3 | Val Acc: 0.9376 | Time: 272s
      🔹 Epoch 3/3 | Val Acc: 0.9395 | Time: 273s
   ✔️ Apollo_Linear seed=777 Final Acc=0.9395

📦 Final results safely archived at: /kaggle/working/ablation_results.zip

✅ NLP ABLATION COMPLETE. Checkpoints, Results and LaTeX tables saved.
